In [1]:
# Décommenter la ligne suivante pour installer les dépendances
#%pip install jupyter_bokeh nbconvert panel watchfiles

# Import librairies et données

In [29]:

import pandas as pd
import panel as pn
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', None)
import numpy as np


import ast
import json


from sklearn.linear_model import RidgeCV, ElasticNetCV, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.metrics import mean_squared_error,mean_absolute_error,mean_absolute_percentage_error,r2_score,root_mean_squared_error

pn.extension()

In [56]:
chemin='C:/Users/marie/Documents/Python/formation_cepe/Projet/'

In [57]:
X_train = pd.read_csv(chemin+'data_projet/X_train.csv')
y_train = pd.read_csv(chemin+'data_projet/y_train.csv')
X_test = pd.read_csv(chemin+'data_projet/X_test.csv')
y_test = pd.read_csv(chemin+'data_projet/y_test.csv')
weights_train = pd.read_csv(chemin+'data_projet/weights_train.csv')
weights_test = pd.read_csv(chemin+'data_projet/weights_test.csv')
resultats = pd.read_csv(chemin+'data_projet/resultats.csv', sep=',')


In [58]:
resultats.columns = ['Modèle', 'RMSE', 'RMSE pondéré', 'MAE', 'MAE pondéré', 'R²', 'R² pondéré', 'RME non optimisé', 'MAE non optimisé', 'R² non optimisé']

In [59]:
X_train=X_train.iloc[:,1:]
X_test=X_test.iloc[:,1:]
y_train=np.array(y_train)[:, 0]
y_test=np.array(y_test)[:, 0]
weights_train=np.log1p(np.array(weights_train)[:, 0])
weights_test=np.log1p(np.array(weights_test)[:, 0])


In [60]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),         # transforme les données
    columns=X_train.columns,               # garde les mêmes noms de colonnes
    index=X_train.index                    # garde les mêmes index
)
X_test_scaled = pd.DataFrame(
    scaler.fit_transform(X_test),         # transforme les données
    columns=X_test.columns,               # garde les mêmes noms de colonnes
    index=X_test.index                    # garde les mêmes index
)

# 1. Introduction

In [ ]:
section_1 = pn.pane.Markdown("""
# 1. Introduction
Dans ce projet, nous avons décidé de nous intéresser à la base de données movies_metatadata.csv trouvable sur Kaggle, qui contient diverses informations sur 45 466 films sortis avant juillet 2017 (https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset). Cette base de données contient les variables suivantes :

- adult : indique si le film est "pour adultes"
- belongs to collection : si le film appartient à une série de films, la variable contient des informations sur la saga
- budget : budget du film
- genres : genres du film
- homepage : lien vers le site officiel du film s'il y en a un
- id : identifiant du film dans la base de données
- imdb_id : identifiant IMDB du film
- original_language : langue originale du film
- original_title : titre original du film
- overview : résumé du film
- popularity : popularité du film sur IMDB
- poster_path : lien vers l'affiche du film
- production_countries : pays de production du film
- production_companies : compagnies/maisons de production du film
- release_date : date de sortie du film
- revenue : recettes du film
- runtime : durée du film
- spoken_languages : langues parlées dans le film en version originale
- status : si le film est sorti, prévu, annulé, en production etc
- tagline : catchphrase du film
- title : titre anglophone du film
- video : False si le film est sorti au cinéma, True s'il est sorti directement sur Internet et qu'il n'a pas été diffusé au cinéma

Nous complétons ces informations avec la base de données credits.csv trouvée au même endroit et qui contient :
- id : identifiant du film dans la base de données
- cast : casting du film 
- crew : équipe du film 

Nous souhaitons à partir de ces données prédire la note de nouveaux films, voir quelles sont les variables les plus décisives pour prédire si un film sera bien noté par le public.
""")



# 2. Retraitements

In [9]:
section_2 = pn.pane.Markdown("""
# 2. Retraitements
Après première exploration des données movies_metadata, nous décidons d'enlever les variables suivantes : 

- adult : elle présente presque toujours la modalité False
- homepage : renseigne le lien vers le site officiel du film s'il y en a un
- overview : contient les résumés des films, ce qui pourrait être intéressant à exploiter mais n'est pas l'objet du projet
- popularity : correspond à la popularité du film sur IMDB au moment où les données ont été extraites. C'est donc une variable qui n'est pas statique et dont nous ignorons le mode de calcul. Nous n'avons pas souhaiter la prendre en compte
- poster_path : lien vers l'affiche du film
- original_language : langue originale du film, extremement corrélée aux pays de production et predominance de l'anglais (71% des films)
- spoken_languages : indique les langues parlées durant le film en version originale, pas utile dans notre projet
- tagline indique la catchphrase du film

Nous allons nous concentrer sur les films qui sont déjà sortis en salle (status = Released et video=False) et qui ont un nombre de vote sur IMDB supérieur strictement à 0. 
Nous enlevons aussi les films où la date de sortie ou l'identifiant IMDB sont manquants (cela représente une soixantaine de films).

La variable revenue est un cas particulier : elle contient les recettes du film mais 84% des observations sont manquantes ou nulles. Par ailleurs, ce n'est pas une donnée statique, en tout cas pour les films récents puisque les recettes augmentent au fur et à mesure que le film se maintient en salle, sort dans de nouveaux pays etc.
C'est une donnée qui n'est d'ailleurs pas du tout disponible pour les films qui viennent de sortir. Pour ces raisons, nous avons décidé d'exclure cette variable. 

Une fois les variables nommées ci-dessus enlevées et les filtres appliqués, la base de données movies_metadata contient 42 028 films et 14 variables.

Les données du fichier credits.csv  permettent d'ajouter les acteurs principaux et le réalisateur du film à notre jeu de données. 


## 2.1. Retraitement de la variable genres

La variable genres, qui indique les différents genres associés au film, est sous forme de liste de dictionnaires. Cela rend un retraitement de cette variable obligatoire.
Chaque film peut avoir entre 0 et 8 genres différents parmi 20 modalités possibles.
Nous les avons regroupé en 10 catégories pour éviter les trop petites classes et limiter le nombre de variables :
* drama
* comedy-family
* thriller-crime,
* romance
* action-adventure
* war-western
* horror-mystery
* science fiction-fantasy
* animation
* intello (documentary, music, history, foreign)

Il existe une modalité "TV movie" qui ne constitue pas un genre en soi mais un support de diffusion. Nous décidons d'ignorer cette modalité et de prendre en compte les autres variables genres associées au film (il y en a toujours au moins une autre).

Nous créons donc 10 variables (une pour chaque catégorie) où chacune d'entre elles vaut 1 si le film appartient à cette catégorie de genres et 0 sinon.
Comme un film peut avoir (et a le plus souvent) plusieurs genres associés, ces indicatrices ne s'excluent pas mutuellement, ce n'est donc pas à proprement parler du "one hot encoding".


## 2.2. Retraitement de la variable belongs_to_collection

Si le film appartient à une saga, cette variable contient les noms des films appartenant à la saga sous forme, là encore, de liste de dictionnaires. 
Cela concerne initialement 4407 films, pour les autres la variable est vide.
Ce format nécessite un retraitement. A partir de "belongs_to_collection" nous créons 3 variables :
- "indic_collec" qui vaut 1 si le film fait partie d'une saga et 0 sinon, 
- "rang" qui indique le rang du film dans la saga (selon sa date de sortie) 
- "vote_prec" qui contient la note du film précédent dans la saga.
REMARQUE : notre périmètre se limite aux films présents dans notre dataset; il se peut que tous les opus d'une série ne soient pas présents.


## 2.3. Retraitement de la variable production_countries

Il s'agit du(des) pays (co)producteur(s) du film sous la forme de liste de dictionnaires.
158 pays distincts sont présents et un même film peut avoir jusqu’à 25 pays coproducteurs.
Attention : les pays sont la plupart du temps triés par ordre alphabétique, cela nous empêche de considérer le pays 1 de la liste comme le pays principal.
Par exemple : le film Le Journal de Bridget Jones, qui est un film produit majoritairement en Angleterre, aurait été considéré comme un film français si nous avions suivi cette logique.

Nous avons donc remplacé les pays par leur continent en distinguant Amérique du Nord et Amérique Latine.
Puis nous avons créé une variable par continent, indiquant le nombre de pays de ce continent associés à chaque film.
Pour reprendre l'exemple du film Le Journal de Bridget Jones : Europe = 3 et North_America = 1.

## 2.4. Retraitement de la variable production_companies

Il s'agit de la ou les maison(s) de production du film sous la forme de liste de dictionnaires.
Un même film peut avoir jusqu’à 26 maisons de production et plus de 20 000 maisons de production distinctes sont présentes dans notre dataset.
Nous souhaitions les regrouper selon leur taille (petites, moyennes, grandes). Pour cela, nous avons considéré comme proxy pour la taille la fréquence d'apparition d'une maison de production dans nos données.
Cela engendre évidemment un biais car c'est très dépendant de nos données de départ.

Pour être rigoureuses, nous avons effectué le comptage des occurrences uniquement sur l'échantillon d'apprentissage. Nous en avons déduit ce découpage en 3 classes :
- les petites : celles  n’apparaissent qu’une fois (13 199)
- les moyennes : celles qui apparaissent entre 2 et 10 fois
- les grosses : + de 10 occurrences
Parmi elles, nous retrouvons les grosses productions américaines dont le record est Warner Bros.(856 films sur l'échantillon train).
Nous créons 3 variables nb_prod_comp_S, nb_prod_comp_M et nb_prod_comp_L qui représentent pour chaque film le nombre de maisons de production petites, moyennes et grandes.

Sur l'échantillon test, si une maison de production était déjà présente dans l'échantillon train, nous lui appliquerons la même taille. 
Si elle n'apparaissait pas dans l'échantillon train, alors nous considérons qu'elle est petite et nous créons les 3 variables nb_prod_comp_S, nb_prod_comp_M, nb_prod_comp_L.

## 2.5. Retraitement de la variable budget
                             
Il s'agit du budget du film exprimé en dollars américains. Dans les données initiales, il y a 33 265 lignes où le budget = 0 (soit près de 80 % des films).
Nous avons tenté de recupérer l'information par webscraping sur Wikipédia.
Les homonymies sur les titres de films ont constitué une complication significative.
Nous avons obtenu finalement 5 868 budgets supplémentaires qu'il a fallu retraiter :
- gestion des séparateurs décimaux différents,
- gestion des montants exprimés en unités ou en millions,
- gestion des fourchettes de montants, 
- conversion des monnaies étrangères, y compris avant l'euro
- conversion en dollars constants de 2017 (avec l'indice d'inflation américain depuis 1960)

Lorsque le budget est inconnu, on lui donne la valeur -1.

## 2.6. Retraitement des variables cast et crew

La variable crew contient les noms de toute l’équipe du film. Elle nous permet de récupérer le nom du réalisateur. De même, la variable cast contient les noms de tous les acteurs du film.
(il peut y avoir jusqu'à 313 acteurs renseignés pour un même film). Nous avons décidé de nous limiter aux 10 acteurs principaux.

Avec ces variables, nous souhaitions savoir si les acteurs principaux et le réalisateur du film étaient déja connus au moment de la sortie du film.
Pour cela, nous avons récupéré par web scraping sur Wikipédia les récompenses obtenues pour chacun des 10 premiers acteurs de chaque film, et leur année d'obtention.

Nous distinguons les "small_awards" (petites récompenses) des "big_awards" (prix internationaux prestigieux).  Nous avons essayé de prendre en compte les récompenses prestigieuses du monde entier (pas uniquement les Oscars et les Césars) afin de ne pas pénaliser les films non occidentaux.
Nous considérons qu'une nomination à une récompense (petite ou grande) vaut autant qu'avoir reçu ce prix.
Ainsi, le nombre de prix que les acteurs/réalisateurs ont gagné ou pour lesquels ils ont été nominés avant l'année de sortie du film peuvent être considérés comme une proxy de leur notoriété de l'époque.
Nous créons 22 variables, correspondant aux nombres de petites et grandes récompenses des 10 acteurs principaux et du réalisateur.

A la fin des retraitements, nous avons 47 variables explicatives.
""")


# 3. Stats descriptives

In [ ]:
section_3 = pn.pane.Markdown("""
# 3. Statistiques descriptives

Notre base de données comportent désormais 42 028 lignes et 47 variables explicatives (+ la variable cible vote_average et la variable vote_count).
                             
Regardons dans un premier temps la distribution de vote_average. 
""")

histo_y = pn.pane.Image(chemin+'images_projet/3.1.histogramme_notes_moyennes.jpeg', width=500)

text_y_hist = pn.pane.Markdown("""
La distribution de vote_average s'approche d'une distribution normale avec une moyenne de 6.09 et un écart-type est de 1.07. Les notes sont donc assez resserrées autour de la moyenne.

Regardons maintenant la distribution du nombre de votes.
""")

histo_vote_count = pn.pane.Image(chemin+'images_projet/3.2.histogramme_vote_count.jpeg', width=500)

text_vote_count_hist = pn.pane.Markdown("""
Le nombre de votes par film prend des valeurs de 1 jusqu'a 14 075 (d'où l'utilité de prendre le logarithme), avec une moyenne de 118,7. 25 % des films ont moins de 5 votes. 

Nous souhaitons garder des films avec un vote_count supérieur à un certain seuil afin d'avoir un peu plus confiance dans les notes moyennes des films. 
Nous allons donc prendre les films avec un vote_count supérieur ou égal à 5, ce qui fait descendre notre base de données à 30 630 lignes. Nous avons choisi de ne pas prendre un seuil plus élevé au risque de se retrouver avec trop peu de données.

Nous nous intéressons maintenant à l'évolution du nombre de films sortis et du nombre de votes à travers les années de sortie des films.
""")

graph_films_votes = pn.pane.Image(chemin+'images_projet/3.3.graph_films_votes.jpeg', width=1000)

texte_gr_films_votes =pn.pane.Markdown("""
On remarque très peu de films avant 1940, à partir de là le nombre de films par an dépasse la centaine et augmente de manière exponentielle à partir de 1990 (création d’IMDB)
NB : le pic de nombre de votes en 1902 est dû au film Le voyage dans la Lune de George Méliès qui a 314 votes.
                                       
En retirant tous les films qui ont moins de 5 votes, nous supposons avoir des notes moyennes plus fiables.

Regardons l'évolution des notes moyennes par année de sortie des films.

""")   

graph_films_notes = pn.pane.Image(chemin+'images_projet/3.4.graph_nb_films_notes_moyennes.jpeg', width=800)

texte_gr_films_notes = pn.pane.Markdown("""
En enlevant les films avec moins de 5 votes, nous disposons de moins de films mais également de notes moins volatiles (avant 1950 notamment).

Contrairement à ce qu’on pourrait croire, en retirant les films avec moins de 5 votes, les notes moyennes augmentent (les films avec peu de votes n’étaient pas notés par le meilleur ami du réalisateur mais plutôt par sa belle-mère).

À partir de 1940 la moyenne des notes est plutôt sur une tendance décroissante (sauf à la fin en 2017).
C'est entièrement possible que les films antérieurs à 1990 qu’on a pris la peine d’ajouter sur IMDB sont ceux déjà devenus cultes, il y aurait donc un biais à la hausse.
Peut-être aussi que les critères de notation et la façon que les gens ont de noter un film n'est pas la même au cours des années (si un film est très apprécié en 2000, a-t-il forcément la même note qu'un film autant apprécié en 2015 ?).

Nous souhaitions au départ prédire la note de films futurs qui n'existent pas encore. 
La notion temporelle était donc à prendre en compte, notamment dans le split train-test en prenant comme données d'entrainement les films plus anciens afin de prédire les films plus récents.
Afin de s'affranchir des biais cités ci-dessus, nous choisissons de changer légèrement notre problématique en considérant que nous allons désormais chercher à prédire les notes de nouveaux films sur IMDB et pas de nouveaux films futurs.
Nous considérons donc que les films sont tous indépendants, même ceux appartenant à une saga car nous avons tenu compte des opus précédents dans les variables explicatives.

Maintenant que nous avons nos données retraitées, nous pouvons regarder les corrélations entre les différentes variables.

""")

correlation = pn.pane.Image(chemin+'images_projet/correlation.png', width=1100)

texte_correlation = pn.pane.Markdown("""
Les variables sont en général peu corrélées, à part indic_collec, rang et vote_prec qui sont 3 variables qui sont renseignées uniquement si le film fait partie d'une saga.
""")

# 4. Machine Learning

In [ ]:
section_4_title = pn.pane.Markdown("""

# 4. Machine Learning

Notre problématique est désormais de prédire la note d'un nouveau film sur IMDB. Ce n'est donc pas forcément un film récent, il peut avoir n'importe quelle année de sortie. 

Notre base de données comporte 30 630 films et 47 variables explicatives. Par ailleurs nous utiliserons vote_count comme poids de pondération dans nos modèles ainsi qu'en pondération des métriques. 

Les données sont séparées en train et test dans les proportions 90% / 10%. Il n'y a pas de valeurs manquantes.

Nous allons tester plusieurs modèles : modèles linéaires, Random Forest, Gradient Boosting et réseau de neurones. Les données ont été standardisées avec StandardScaler pour les modèles linéaires et le réseau de neurones.

Nous nous intéressons particulièrement aux métriques Root Mean Squared Error (RMSE), Mean Absolute Error (MAE) et R², le critère principal étant le RMSE. 
Nous regardons également les métriques pondérées afin de savoir si nous arrivons à mieux prédire les films populaires.

## 4.1 Modèles linéaires

### 4.1.1 Ordinary Least Squares et Weighted Least Squares

Le premier modèle OLS sert de benchmark, avec les 47 variables explicatives. Pour le modèle WLS, l’idée est de tenir compte du fait qu’une note moyenne basée sur 5 avis ne devrait pas avoir le même poids dans notre apprentissage qu’une note basée sur 10000 avis.
Ainsi nous pondérons nos observations d’apprentissage (X_train) par log(vote_count).
                                 """)

resultats_wls = pn.pane.DataFrame(resultats[resultats["Modèle"].isin(['OLS', 'WLS'])].iloc[:,0:7])

section_4_after_wls = pn.pane.Markdown("""Le RMSE et le MAE sont plûtot bons, cependant le R² reste relativement faible. 
Les métriques pondérées sont meilleures que les métriques classiques, ce qui est plutôt bon signe. Les métriques pour le modèle OLS sont meilleures que pour le modèle WLS, ce qui est inattendu. Regardons maintenant les résidus vs prédictions du modèle OLS :""")

residus_1 = pn.pane.Image(chemin+'images_projet/residus_1.jpg', width=500)

section_4_after_residus_1 = pn.pane.Markdown("""Les résidus sont en forme de boule, ce qui suggère que la variance des résidus n'est pas constante (hétéroscédasticité). 
Nous allons examiner les résidus selon certaines variables explicatives pour voir si une transformation de variable peut être intéressante.""")

residus_2 = pn.pane.Image(chemin+'images_projet/residus_2.jpg', width=500)

section_4_after_residus_2 = pn.pane.Markdown("""Les résidus en fonction de la variable release_year sont en forme de cône, il y a donc un problème d'hétéroscédasticité lié à cette variable. 
C'est également le cas pour les variables runtime et budget (graphiques non présents ici). Nous allons appliquer la fonction logarithme à ces variables et observer s'il y a un changement dans les résidus. 
Nous avons également regardé les résidus en fonction des autres variables (avec des boxplots pour des variables binaires ou avec peu de modalités) et aucune autre ne semble intéressante à transformer.

### 4.1.2 OLS avec transformations log

Regardons les métriques avec ce nouveau modèle ainsi que les résidus.""")

resultats_wls_log = pn.pane.DataFrame(resultats[resultats["Modèle"].isin(['OLS', 'OLS transformation log'])].iloc[:,0:7])

residus_3 =pn.pane.Image(chemin+'images_projet/residus_3.jpg', width=500)

section_4_after_residus_3 = pn.pane.Markdown("""Les métriques pour le modèle OLS avec transformations log sont moins bonnes que pour le modèe OLS simple, et les résidus ont la même forme de boule.
Après avoir regardé les graphiques des différentes variables explicatives avec la variable cible, la relation linéaire n'est pas évidente. 
Notre hypothèse est donc que les modèles linéaires simples sont insuffisants pour capter les relations entre nos variables explicatives et la note moyenne des films.

C'est pourquoi nous allons désormais tester des modèles non linéaires.""")

section_4_rf = pn.pane.Markdown("""
## 4.2 Random Forest

Comme nous l'avons vu précédemment, le RMSE et le MAE sont plûtot bons, cependant le R² reste relativement faible. 
Etant donné que la distribution de la variable cible correspond à celle d'une loi normale, on peut supposer que le modèle prédit en moyenne correctement (bon RMSE) mais ne capte pas bien la variance des données (notes extrêmes).
Dans l'optimisation des hyperparamètres du modèle, nous allons donc chercher à maximiser le R² et regarder si cela améliore ou détériore le RMSE. Les observations d’apprentissage (X_train) sont pondérées par log(vote_count).

Les hyperparamètres que nous avons optimisés sont les suivants : 
- n_estimators : de 100 à 1000
- max_depth : de None à 100
- min_samples_split : de 2 à 10
- min_samples_leaf : de 1 à 4
- max_features : sqrt ou log2

L'optimisation des hyperparamètres s'est faite par cross validation sur 5 folds.

Le modèle Random Forest optimal est celui avec n_estimators=1000, max_depth = 50, min_samples_split=2, min_samples_leaf=2 et max_features=sqrt.
Ci-dessous le tableau des métriques classiques et pondérées pour le Random Forest :""")

resultats_rf = pn.pane.DataFrame(resultats[resultats["Modèle"].isin(['Random Forest'])].iloc[:,0:7])

section_4_gb=pn.pane.Markdown("""


Nous pouvons observer que toutes les métriques sont meilleures que celles des modèles linéaires. 
Les RMSE et MAE (classiques et pondérés) ont augmenté avec le R², ce qui indique que le modèle a réussi à mieux capturer la variance des données et à prédire plus finement les notes.
Les métriques pondérées sont toujours meilleures que les métriques classiques, ce qui signifie que le modèle est meilleur sur les films avec le plus de votes, donc plus populaires.

## 4.3 Gradient Boosting

Les observations d’apprentissage (X_train) sont pondérées par log(vote_count)
Dans l'optimisation des hyperparamètres du modèle, nous allons chercher à maximiser le R² avec la même logique que pour le Random Forest.

Les hyperparamètres que nous avons optimisés sont les suivants : 
- n_estimators : de 100 à 1000
- learning rate : de 0.01 à 0.1
- max_depth : de 3 à 10
- min_samples_leaf : de 1 à 4
- subsample : de 0.8 à 1

L'optimisation des hyperparamètres s'est faite par cross validation sur 5 folds.

Le modèle Gradient Boosting optimal est celui avec n_estimators=1000, learning_rate=0.01, max_depth = 10, min_samples_leaf=3 et subsample=0.8.
Ci-dessous le tableau des métriques classiques et pondérées pour le Gradient Boosting :

""")

resultats_gb = pn.pane.DataFrame(resultats[resultats["Modèle"].isin(['Gradient Boosting'])].iloc[:,0:7])

section_4_nn = pn.pane.Markdown(""" Les métriques du Gradient Boosting sont meilleures que celles du Random Forest. Les métriques pondérées sont toujours meilleures que les métriques classiques.
Nous allons désormais tester un réseau de neurones.

## 4.4 Réseaux de neurones

Nous estimons avoir assez de données pour essayer un réseau de neurones simple. Nous avons testé un DNN fully-connected avec les caractéristiques suivantes :
- une couche d'entrée de dimension 47
- deux couches cachées denses avec relu 
- mécanisme de régularisation par dropout après chaque couche cachée
- une couche de sortie linéaire de dimension 1
- callback Early stopping avec la métrique R² avec une patience de 20
- optimiseur Adam
- fonction de perte MAE
- un batch size de 64
- un nombre d'epochs maximum de 200
Les données d'entraînement ont été séparées en deux sous-ensembles dans les proportions 80/20: un pour l'entraînement du DNN et l'autre pour la validation dans le cadre de l'EarlyStopping.
                                """)

resultats_nn = pn.pane.DataFrame(resultats[resultats["Modèle"].isin(['Reseau de neurones'])].iloc[:,0:7])

section_4_after_nn = pn.pane.Markdown(""" Les métriques du réseau de neurones ne sont pas très satisfaisantes (meilleures que les modèles linéaires mais moins bonnes que le Random Forest ou le Gradient Boosting). 
Quelques configurations différentes ont été essayées sans grande différence dans les résultats.

Nous pouvons donc conclure que le modèle prédisant le mieux les notes de nouveaux films sur IMDB est le Gradient Boosting. A noter que le Gradient Boosting se trompe en moyenne de 0.69 points mais environ 35% de la variabilité des données est expliquée par ce modèle.
                                """)

# 5. Conclusion

In [ ]:
section_5_1 = pn.pane.Markdown("""
# 5. Conclusion
Notre problématique est désormais de prédire la note d'un nouveau film sur IMDB grâce à plusieurs variables explicatives relatives aux caractéristiques du film (durée, budget, genres, pays et maisons de production, acteurs et réalisateur).

Nous avons essayé plusieurs modèles avec pondération des observations d'entraînement avec la variable vote_count. Ci-dessous le tableau récapitulatif des métriques RMSE, MAE et R² (classiques et pondérées) :
""")

resultats_globaux = pn.pane.DataFrame(resultats.iloc[:,0:7])

section_5_2 = pn.pane.Markdown("""
D'après nos métriques, c'est le Gradient Boosting qui est le plus performant pour prédire les notes.
L'importance des variables via la permutation importance nous permet d'observer le RMSE (non pondéré) en fonction du nombre de variables dans le modèle (classées par ordre d'importance):
""")

permutation =pn.pane.Image(chemin+'images_projet/permutation_importance.jpg', width=500)

section_5_3 = pn.pane.Markdown("""
Le RMSE a tendance à diminuer au fur et à mesure qu'on ajoute des variables explicatives, ce qui signifie qu'aucune n'est à enlever.       
Les 5 variables avec le plus d'importance sont : runtime, release_year, Intello, Drama et budget.

Bien sûr il y a de nombreuses limites à cette étude :
- le choix des variables explicatives : nous aurions pu modifier différemment les variables déjà présentes dans le dataset ou ajouter des variables différentes  
- l'exactitude du web scraping : Wikipédia n'étant pas entièrement standardisé, les informations récupérées peuvent être inexactes ou incomplètes
- les retraitements sur la variable budget : les plus gros budgets ( en milliards de dollars !) sont essentiellement des très vieux films pour lesquels les conversions de monnaies et le retraitement pour passer en dollars américains de 2017 sont sans doute inadaptés (taux de conversion actuel vs conversion à l’époque, inflation disponible seulement à partir de 1960…)
- le choix du seuil de vote_count à partir duquel on considère que la moyenne des votes est fiable : nous aurions pu choisir un seuil supérieur à 5 votes mais c’était un compromis pour ne pas perdre trop d’observations
- une optimisation différente des modèles : la principale contrainte a été le temps, nous souhaitions que le tuning des hyperparamètres puisse être assez rapide
- ...
""")








# 6. Exploration modèle

In [ ]:
X_train = pd.read_csv(chemin+'data_projet\X_train.csv')
y_train = pd.read_csv(chemin+'data_projet\y_train.csv')
X_test = pd.read_csv(chemin+'data_projet\X_test.csv')
y_test = pd.read_csv(chemin+'data_projet\y_test.csv')
weights_train = pd.read_csv(chemin+'data_projet\weights_train.csv')
weights_test = pd.read_csv(chemin+'data_projet\weights_test.csv')
joker = pd.read_csv(chemin+'data_projet\dataframe_joker.csv')

X=pd.concat([X_train,X_test], axis=0).reset_index()
y=pd.concat([y_train,y_test], axis=0).reset_index()
y=y.iloc[:,1:]
weights = pd.concat([weights_train,weights_test], axis=0)



,vote_average
0,6.9
1,5.3
2,7.8
3,7.0
4,7.8
...,...
30625,5.3
30626,5.5
30627,5.6
30628,5.0


In [42]:
X=X.iloc[:,1:]
X=X.iloc[:,1:]
y=np.array(y)[:, 0]
weights=np.log1p(np.array(weights)[:, 0])

In [ ]:
# ---- 1. Modèle ML ----
model = GradientBoostingRegressor(random_state=42, learning_rate= 0.01, max_depth= 10, min_samples_leaf= 3, n_estimators= 1000, subsample= 0.8).fit(X, y, sample_weight=weights)

# ---- 2. Widgets ----
wg_runtime = pn.widgets.FloatSlider(name="runtime", start=0, end=300, step=1, value=138)
wg_indic_collec = pn.widgets.FloatSlider(name="indic_collec", start=0, end=1, step=1, value=1)
wg_rang = pn.widgets.FloatSlider(name="rang", start=0, end=25, step=1, value=1)
wg_vote_prec = pn.widgets.FloatSlider(name="vote_prec", start=0, end=10, step=0.1, value=8.3)
wg_year = pn.widgets.FloatSlider(name="release_year", start=1850, end=2050, step=1, value=2024)
wg_intello =pn.widgets.FloatSlider(name="Intello", start=0, end=1, step=1, value=1)
wg_drama=pn.widgets.FloatSlider(name="Drama", start=0, end=1, step=1, value=1)
wg_budget=pn.widgets.FloatSlider(name="budget", start=0, end=1000000000, step=1000000, value=300000000)


# ---- 3. Fonction liée aux widgets ----
def predict(x1_val, x2_val, x3_val, x4_val,x5_val, x6_val, x7_val, x8_val):
    pred = model.predict([[x1_val,x2_val,x3_val,x4_val, x5_val,1,0,0,0,0,0,x6_val,0,1,0,0,0,0,0,0,x7_val,4,1,2,21,97,38,49,6,16,4,43,0,4,7,7,0,0,0,0,0,0,0,3,3,18, x8_val]])[0]
    return pn.pane.Markdown(f"## Prédiction du modèle : **{pred:.3f}**")

prediction_panel = pn.bind(predict, x1_val=wg_runtime, x2_val=wg_indic_collec, x3_val=wg_rang, x4_val=wg_vote_prec, x5_val=wg_year, x6_val=wg_drama, x7_val=wg_intello, x8_val=wg_budget)

# ---- 4. Interface ----
section_6= pn.Column(
    "# 6. Explorez le modèle de régression",
    wg_runtime,
    wg_indic_collec,
    wg_rang,
    wg_vote_prec,
    wg_year,
    wg_drama,
    wg_intello,
    wg_budget,
    prediction_panel,
)

In [ ]:
# ---- Zone centrale qui change ----
main_area = pn.Column(section_1, sizing_mode="stretch_width")

# ---- Fonctions de navigation ----
def show_section_1(event):
    main_area.objects = [section_1]

def show_section_2(event):
    main_area.objects = [section_2]

def show_section_3(event):
    main_area.objects = [section_3,
        histo_y,
        text_y_hist,
        histo_vote_count,
        text_vote_count_hist,
        graph_films_votes,
        texte_gr_films_votes,
        graph_films_notes,
        texte_gr_films_notes,
        correlation]

def show_section_4(event):
    main_area.objects = [section_4_title,
                         resultats_wls, 
                         section_4_after_wls,
                         residus_1,
                         section_4_after_residus_1,
                         residus_2,
                         section_4_after_residus_2,
                         resultats_wls_log,
                         residus_3,
                         section_4_after_residus_3,
                         section_4_rf,
                         resultats_rf,
                         section_4_gb,
                         resultats_gb,
                         section_4_nn,
                         resultats_nn,
                         section_4_after_nn]

def show_section_5(event):
    main_area.objects = [section_5_1, resultats_globaux, section_5_2, permutation, section_5_3]

def show_section_6(event):
    main_area.objects = [section_6]
# ---- Sidebar = menu latéral ----
sidebar = pn.Column(
    pn.pane.Markdown("## Menu"),
    pn.widgets.Button(name="1. Introduction"),
    pn.widgets.Button(name="2. Retraitements"),
    pn.widgets.Button(name="3. Statistiques descriptives"),
    pn.widgets.Button(name="4. Machine Learning"),
    pn.widgets.Button(name="5. Conclusion"),
    pn.widgets.Button(name="6. Explorez le modèle de régression")
)

# Association des boutons aux fonctions
sidebar[1].on_click(show_section_1)
sidebar[2].on_click(show_section_2)
sidebar[3].on_click(show_section_3)
sidebar[4].on_click(show_section_4)
sidebar[5].on_click(show_section_5)
sidebar[6].on_click(show_section_6)

# ---- Template global ----
template = pn.template.MaterialTemplate(title="Projet Datascience - Cécile Prévot et Marie Le Pennec")

template.sidebar.append(sidebar)
template.main.append(main_area)

template.servable()


MaterialTemplate
    [js_area] HTML(None, height=0, margin=0, sizing_mode='fixed', width=0)
    [actions] MaterialTemplateActions()
    [browser_info] BrowserInfo()
    [busy_indicator] LoadingSpinner(height=20, width=20)
    [nav-2598324730512] Column
        [0] Markdown(str)
        [1] Button(name='1. Introduction')
        [2] Button(name='2. Retraitements')
        [3] Button(name='3. Statistiques d...)
        [4] Button(name='4. Machine Learning')
        [5] Button(name='5. Conclusion')
    [main-2598324545680] Column(sizing_mode='stretch_width')
        [0] Markdown(str)

In [ ]:
#panel serve --autoreload --show --allow-websocket-origin=$(echo $VSCODE_PROXY_URI | cut -d '/' -f 3) /home/onyxia/work/formation_cepe/Projet/panel_projet.ipynb